In [1]:
!pip install git+https://github.com/huggingface/transformers@v4.49.0-SmolVLM-2

from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

  Cloning https://github.com/huggingface/transformers (to revision v4.49.0-SmolVLM-2) to /private/var/folders/7x/0pnrdfvn0ts5vvq0s97bwpl80000gn/T/pip-req-build-0otf7jj4
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /private/var/folders/7x/0pnrdfvn0ts5vvq0s97bwpl80000gn/T/pip-req-build-0otf7jj4
  Running command git checkout -q 61e3ffd8148e68d879e3b2e1609fbb7d99621276
  Resolved https://github.com/huggingface/transformers to commit 61e3ffd8148e68d879e3b2e1609fbb7d99621276
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
model_path = "HuggingFaceTB/SmolVLM2-256M-Video-Instruct"

processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    _attn_implementation="flash_attention_2"
).to("cuda")


In [ ]:
PROMPT = (
    "Describe all the key events in the entire video. "
)

SYSTEM = (
    "Focus only on describing the key dramatic action or notable event occurring in this video segment. "
    "Skip general context or scene-setting details unless they are crucial to understanding the main action. "
)

messages = [
    {
        "role": "system",
        "content": SYSTEM
    },
    {
        "role": "user",
        "content": [
            {"type": "video", "path": "TVSum/video_5.mp4"},
            {"type": "text", "text": PROMPT}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=500)
print(processor.batch_decode(generated_ids, skip_special_tokens=True)[0])

RuntimeError: Mismatched Tensor types in NNPack convolutionOutput